# Queen Editor — Bağlantı (Bölüm 2)

Repoyu klonlar → **Flask** derlenmiş arayüzü (`frontend/dist`) servis eder → **cloudflared** linki basar.
Açılan sayfa sunucuya `/api/health` isteği atıp **"sunucuya bağlı ✓"** gösterir. Drive, ComfyUI yok.

Arayüz repoya **derlenmiş** gelir (ComfyUI deseni: frontend geliştiricide derlenir, Colab'da npm/build
çalışmaz). Colab yalnız klonlar ve servis eder.

## Kullanım
1. Bu `app.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. Colab'ın **🔑 Secrets** panelinde `GITHUB_TOKEN` ekli olmalı (fine-grained, yalnız bu repo,
   `Contents: read` — kurulum için `README.md`). Bir kez eklenir, sonra hazır.
3. **Runtime → Run all** → en alttaki linke gir → **"sunucuya bağlı ✓"** görmelisin.

In [ ]:
# === CONFIG ===
# The GitHub token comes from Colab's Secrets store (🔑 in the left sidebar), NOT this cell
# -- set once per Google account, never pasted again, never in the notebook source or git.
# Add a secret named GITHUB_TOKEN (fine-grained, this repo, "Contents: read") and grant this
# notebook access. See README for the token setup.
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""   # secret missing or access not granted -> the assert below explains the fix

BRANCH    = "feat/queen-editor-v1"       # dev branch for now; switch to "main" after merge
REPO      = "AltanBaysal/Internal-tools" # <owner>/<repo>
CLONE_DIR = "/content/Internal-tools"    # clone target on Colab's local disk
APP_PORT  = 8000                         # Flask port (matches backend/config.py)

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu notebook'a erişimi aç (fine-grained, yalnız bu repo, Contents: read)."
)
print("✓ CONFIG hazır (token Colab Secrets'tan okundu)")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")

In [ ]:
# === Clone (delete-and-reclone: the local tree is disposable, always fetch the latest) ===
# subprocess.run with an argument LIST (not shell=True): the token never reaches the shell
# history or a log line. On failure git's stderr is printed RAW, with the token masked.
import os, shutil, subprocess

def _mask(text):
    """Replace the token with <token> so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)              # no pull/merge -- a fresh clone has one behaviour

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"   # never printed (carries the token)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Raw git output, token masked -- never invent a cause (repo comment rule).
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The built frontend ships in the repo (frontend/dist) -- fail loud if it is missing, so a
# forgotten rebuild-and-commit shows up here, not as a blank page.
DIST = os.path.join(CLONE_DIR, "queen-editor", "frontend", "dist", "index.html")
assert os.path.exists(DIST), f"❌ Derlenmiş arayüz yok: {DIST} — frontend'i derleyip commit'le (README)"
print("✓ Klon tamam (derlenmiş arayüz mevcut)")

In [ ]:
# === Start Flask (background) + cloudflared tunnel ===
# Flask serves the pre-built frontend/dist and /api. It runs as a module from queen-editor/ so
# `backend` resolves as a package. The cell stays OPEN (tail -f): if it ends, Colab calls the
# runtime idle and kills the tunnel. No npm/build here -- the UI ships built (ComfyUI pattern).
import subprocess, time, os, re, urllib.request

APP_DIR = os.path.join(CLONE_DIR, "queen-editor")
FLASK_LOG = "/content/flask.log"

# Re-run safety: kill previous instances before starting new ones
subprocess.run(["pkill", "-f", "backend.main"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(FLASK_LOG, "w")
subprocess.Popen(["python", "-m", "backend.main"], cwd=APP_DIR, stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(FLASK_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Flask 90 sn içinde /api/health'e cevap vermedi — yukarıdaki log'a bak")
print(f"✓ Flask ayakta ({(i + 1) * 2}s)")

if not os.path.isfile("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunlog):
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read())
        if m:
            link = m.group(0)
            break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 Queen Editor: {link}\n")
print("⬆️  Linke gir → 'sunucuya bağlı ✓' görmelisin.\n")
print("📡 Sunucu çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", FLASK_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — Flask hâlâ arka planda (yeni link için tekrar çalıştır).")